In [15]:
import torch
import myneural as mn
import torch.nn.functional as F
import time
import pyarrow.parquet as pq
import sentencepiece as spm
import math
torch.manual_seed(42)

In [16]:
# training tokenizer --------------
TRAIN = False
if TRAIN:
    pf = pq.ParquetFile('data/tinystories0.parquet')
    with open('data/input.txt', 'w', encoding='utf-8') as f:
        for i in range(100):
            text_list = pf.read_row_group(i).column('text').to_pylist() # 1000 rows each row group     
            for story in text_list:
                f.write(story)
                
    spm.SentencePieceTrainer.train( # pyright: ignore[reportAttributeAccessIssue]
        input = 'data/input.txt',
        model_prefix = 'models/token_model',
        vocab_size = 4096,
        model_type = 'bpe',
        character_coverage = 1.0,
        byte_fallback = True,
        add_dummy_prefix = True,
        )

In [17]:
sp = spm.SentencePieceProcessor(model_file="models/token_model.model") # pyright: ignore[reportCallIssue]
sp.Encode("Lily found a needle.") # pyright: ignore[reportAttributeAccessIssue]
print(''.join(sp.IdToPiece(id) for id in sp.Encode("Bob saw a shiny golden penny.\n He liked it.") + [sp.eos_id()] + sp.Encode("Lily found a flower.")).replace('▁', ' ').replace('</s>', '</s>\n')) # EOS IDs are hidden in general.

 Bob saw a shiny golden penny. He liked it.</s>
 Lily found a flower.


In [18]:
config = mn.Config()

In [ ]:
config.n_embd = 256
config.n_head = 8
config.n_layer = 8
config.mini_batch_size = 128 # B
config.context_length = 256 # T
config.total_batch_size = 1 * config.mini_batch_size * config.context_length # MUST BE DIVISBLE by (B*T)
config.device = 'cuda'
config.vocab_size = 4096

max_iter = 6000
B = config.mini_batch_size
T = config.context_length
grad_accum_steps = config.total_batch_size // (B*T)
assert config.total_batch_size % (B * T) == 0, "Total batch size must be divisble by (B*T)"

In [ ]:
import dataset
set = dataset.ParquetDataset([
    'data/tinystories0.parquet',
    ],
    B = config.mini_batch_size,
    T= config.context_length,
    tokenizer=sp,
    append_eos=True,
    loop_around=True,
    shuffle_rgs=True,
    seed=42,
)

dataloader = torch.utils.data.DataLoader(
    set,
    batch_size=None,# batch_size = None because i already give (B, T) in the dataset.
    # if i gave a batch_size, it would give me (batch_size, B, T)
    num_workers=6,
    pin_memory=True,
    prefetch_factor=4, # 2 batches will be loaded ahead of time.
    persistent_workers=True
) 

In [21]:
val_set = dataset.ParquetDataset([
    'data/tinystories0val.parquet',
    ],
    B = config.mini_batch_size,
    T= config.context_length,
    tokenizer=sp,
    append_eos=True,
    loop_around=False,
    shuffle_rgs=False,
    seed=42,
)

val_dataloader = torch.utils.data.DataLoader(
    val_set,
    batch_size=None,
    num_workers=0,
    pin_memory=True,
) 

In [22]:
def get_lr(step):
    max_lr = 1e-3
    min_lr = 1e-4
    warmup_steps=200
    max_steps=6000
    if step < warmup_steps:
        return max_lr * (step+1)/warmup_steps 
    if step > max_steps:
        return min_lr
    decay_ratio = (step-warmup_steps)/(max_steps-warmup_steps)
    coefficient = 0.5 * (1.0 + math.cos(decay_ratio * math.pi)) # starts at 1 reduces to 0
    return min_lr + coefficient * (max_lr - min_lr)
    

In [23]:
torch.set_float32_matmul_precision('high')
model = mn.Transformer(config, dtype=torch.float32, device=config.device)
params = model.parameters()

In [24]:
optimizer = model.configure_optimizer(weight_decay=0.1, learning_rate=get_lr(0), device=config.device)

In [25]:
# # Load the list of tensors
# loaded_params = torch.load("models/params_new.pth")

# # Manually re-assign them to your model
# for p, loaded_p in zip(model.parameters(), loaded_params):
#     p.data.copy_(loaded_p.data)

In [26]:
model_optim = torch.compile(model)
step = 0

In [27]:
def get_val_loss():
    max_batches = 25
    val_loss = []
    with torch.no_grad():
        for i, (xs, ys) in enumerate(val_dataloader):      
            if i >= max_batches:
                break
            xs = xs.to(device=config.device, non_blocking=True)
            ys= ys.to(device=config.device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                _, loss = model_optim(xs, targets=ys)
            val_loss.append(loss.item())
    return sum(val_loss)/len(val_loss)

In [ ]:
data_iter = iter(dataloader)
lossi = []
timei = []
mean_stat_batches = 25
print_batches = 100
val_batches = 300
for _ in range(step, max_iter):
    t_start = time.time()
    optimizer.zero_grad(set_to_none=True)
    # forward pass
    loss_accum = torch.tensor(0.0, device=config.device)
    for micro_step in range(grad_accum_steps):
        xs, ys= next(data_iter) # loop_around=True so we don't need to worry about StopIteration exception
        xs = xs.to(device=config.device, non_blocking=True)
        ys= ys.to(device=config.device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits, loss = model_optim(xs, targets=ys)
        loss = loss / grad_accum_steps # normalizer to avoid loss gradients being exagerrated by grad_accum_steps times.
        loss_accum += loss.detach()
        # backward pass
        loss.backward()
    
    norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
    
    # update
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    optimizer.step()
    
    torch.cuda.synchronize()
    t_end = time.time()
    
    # track stats    
    lossi.append(loss_accum.item())
    timei.append(t_end-t_start)
    if step % print_batches == 0:
        lossi = lossi[-mean_stat_batches:]
        timei = timei[-mean_stat_batches:]
        train_loss = sum(lossi)/len(lossi)
        if step % val_batches == 0:
            val_loss = get_val_loss()
        else: 
            val_loss = 0.0
        dt = sum(timei)/len(timei)
        tokens_per_sec = (xs.shape[0] * xs.shape[1] * grad_accum_steps)/dt # (B*T)/dt
        print(f"Step {step}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | dt: {dt*1000:.2f}ms | {int(tokens_per_sec):,d} tok/s")

    step += 1

Step 0: Train Loss: 8.8235 | Val Loss: 8.7507 | dt: 2106.18ms | 31,116 tok/s
Step 100: Train Loss: 8.3240 | Val Loss: 0.0000 | dt: 252.58ms | 259,469 tok/s
Step 200: Train Loss: 8.3280 | Val Loss: 0.0000 | dt: 293.81ms | 223,055 tok/s


KeyboardInterrupt: 

In [ ]:
val_loss = get_val_loss()

In [ ]:
SAVE = False
if SAVE:
    params_list = list(model.parameters())
    torch.save(params_list, "models/params_1.pth")

In [ ]:
mn.prompt("", model=model, tokenizer=sp, new_tokens=256, debug=True, device=config.device)

<|E|> One day, a boy named Terry and his mom had a great idea. They decided to go to the restaurant maker. Terry was ordinary. He was excited to go shopping through the computers and pay to his destination. When they arrived, Terry was so excited! He quickly grabbed a handful of them and took one. He watched as the orderer crer fixed his ear and flew off to the top with the pictures. It was the perfect way to stay. Everyone was so happy as the food was sent to that paste! They danced and played together all day. When it was time to go home, Terry didn't want to leave. Suddenly, he heard an urge of farm. He hopped away to find another place in the big field! Terry was so happy, he jumped up and gave the people a big welcome. Everyone was so surprised to see the sandwiches and the other chickens home in its mind. Everyone was very happy, and they were never envious of the new toys!<|E|> Bob went to visit the chubby creeker. He was very excited. The creeker showed him lots of interesting 